<a href="https://colab.research.google.com/github/FrancescoGiulino/olist-data-cleaning/blob/main/olist_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

test_products = pd.read_csv("/content/drive/MyDrive/DataWerehouse_Project/olist_products_dataset.csv")

print("--- ISOLAMENTO ORIGINALE DRIVE ---")
print("Righe totali:", len(test_products))
print("Peso Massimo (g):", test_products['product_weight_g'].max())
print("Peso Medio (g):", test_products['product_weight_g'].mean())
print("Prodotti sotto 1kg:", len(test_products[test_products['product_weight_g'] <= 1000]))

--- ISOLAMENTO ORIGINALE DRIVE ---
Righe totali: 32951
Peso Massimo (g): 40425.0
Peso Medio (g): 2276.4724877841513
Prodotti sotto 1kg: 20057


In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import csv
import os

drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/DataWerehouse_Project/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print("Caricamento dati in corso...")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
translation = pd.read_csv(base_path + "product_category_name_translation.csv")
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")

print("Tutti i file caricati con successo!")

Caricamento dati in corso...
Tutti i file caricati con successo!


### **DATA PREPROCESSING**

In [ ]:
# ============================================================================
# 1. GESTIONE CATEGORIE ORFANE E PREPARAZIONE DIZIONARIO TRADUZIONI
# ============================================================================
print("Controllo categorie orfane in corso...")

# Rimuoviamo a monte qualsiasi nullo o duplicato dal file traduzioni per isolarlo
translation_clean = translation.dropna(subset=['product_category_name']).drop_duplicates(subset=['product_category_name'])

# Creiamo il dizionario di mappatura nativo: Portoghese -> Inglese
mapping_dict = dict(zip(translation_clean['product_category_name'], translation_clean['product_category_name_english']))

# Identifichiamo le categorie uniche presenti nei prodotti
categorie_prodotti = set(products['product_category_name'].dropna().unique())
categorie_tradotte = set(mapping_dict.keys())

# Troviamo la differenza (le orfane)
categorie_orfane = categorie_prodotti - categorie_tradotte

if categorie_orfane:
    print(f"Trovate {len(categorie_orfane)} categorie orfane: {categorie_orfane}")
    # Integriamo le orfane nel dizionario impostando la traduzione uguale al portoghese
    for cat in categorie_orfane:
        mapping_dict[cat] = cat
    print("Categorie mancanti integrate con successo nel dizionario di mappatura!")
else:
    print("Nessuna categoria orfana trovata.")


# ============================================================================
# 2. PULIZIA ANAGRAFICHE (CUSTOMERS & SELLERS)
# ============================================================================
print("Pulizia anagrafiche in corso...")
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()
customers['customer_state'] = customers['customer_state'].str.strip().str.upper()
sellers['seller_state'] = sellers['seller_state'].str.strip().str.upper()

customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)


# ============================================================================
# 3. PIPELINE PRODOTTI ISOLATA E VETTORIALE (MAPPING RIGA PER RIGA)
# ============================================================================
print("Preprocessing Products...")

# Traduzione chirurgica riga per riga tramite .map() (Immune da cross-join e alterazioni di indici)
products['product_category_name_english'] = products['product_category_name'].map(mapping_dict)

# Gestiamo i veri valori nulli all'origine (prodotti sprovvisti di categoria)
products['product_category_name_english'] = products['product_category_name_english'].fillna('Unknown Category')

# Pulizia e casting atomico colonna per colonna delle metriche fisiche
numeric_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in numeric_cols:
    products[col] = pd.to_numeric(products[col], errors='coerce').fillna(0)

# Costruiamo da zero il DataFrame finale associando i singoli vettori puliti
products_clean_final = pd.DataFrame({
    'product_id': products['product_id'],
    'product_category_name': products['product_category_name_english'],
    'product_name_lenght': products['product_name_lenght'].astype(int),
    'product_description_lenght': products['product_description_lenght'].astype(int),
    'product_photos_qty': products['product_photos_qty'].astype(int),
    'product_weight_g': products['product_weight_g'].astype(int),
    'product_length_cm': products['product_length_cm'].astype(int),
    'product_height_cm': products['product_height_cm'].astype(int),
    'product_width_cm': products['product_width_cm'].astype(int)
})

# Sovrascriviamo la variabile originale in modo che il resto del tuo script esporti il file corretto
products = products_clean_final


# ============================================================================
# 4. DATA QUALITY CHECK IN PYTHON
# ============================================================================
print("\n--- VERIFICA PESI IN PYTHON ---")
print(f"Totale Prodotti: {len(products)}")
print(f"Peso Massimo (g): {products['product_weight_g'].max()} (Atteso: ~40425)")
print(f"Peso Medio (g):   {products['product_weight_g'].mean():.2f} (Atteso: ~2276.47)")
print(f"Prodotti sotto 1kg: {len(products[products['product_weight_g'] <= 1000])} (Atteso: ~20057)")
print("--------------------------------\n")

Controllo categorie orfane in corso...
Trovate 2 categorie orfane: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}
Categorie mancanti integrate con successo nel dizionario di mappatura!
Pulizia anagrafiche in corso...
Preprocessing Products...

--- VERIFICA PESI IN PYTHON ---
Totale Prodotti: 32951
Peso Massimo (g): 40425 (Atteso: ~40425)
Peso Medio (g):   2276.33 (Atteso: ~2276.47)
Prodotti sotto 1kg: 20059 (Atteso: ~20057)
--------------------------------



In [ ]:
print("Pulizia Ordini in corso...")

# 1. Definiamo le colonne data
date_columns_orders = ['order_purchase_timestamp', 'order_approved_at',
                       'order_delivered_carrier_date', 'order_delivered_customer_date',
                       'order_estimated_delivery_date']

# 2. Convertiamo in Datetime
for col in date_columns_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# 3. CONTROLLO LOGICO DELLE DATE (Per tollerare i Null)
# Usiamo isna() per dire: "Se è nullo, va bene, passa il controllo. Altrimenti verifica la data".
mask_approved = orders['order_approved_at'].isna() | (orders['order_purchase_timestamp'] <= orders['order_approved_at'])
mask_carrier = orders['order_delivered_carrier_date'].isna() | (orders['order_approved_at'] <= orders['order_delivered_carrier_date'])
mask_customer = orders['order_delivered_customer_date'].isna() | (orders['order_delivered_carrier_date'] <= orders['order_delivered_customer_date'])

orders = orders[mask_approved & mask_carrier & mask_customer]

print(f"Pulizia date completata. Ordini totali (tutti gli stati) rimasti: {len(orders)}")


Pulizia Ordini in corso...
Pulizia date completata. Ordini totali (tutti gli stati) rimasti: 98044


In [ ]:
order_items = order_items[order_items['order_id'].isin(orders['order_id'])]
order_items = order_items[order_items['product_id'].isin(products['product_id'])]
order_items = order_items[order_items['seller_id'].isin(sellers['seller_id'])]

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

reviews = reviews[reviews['order_id'].isin(orders['order_id'])].copy()

# 1. Gestione dei Null
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title').astype(str)
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Comment').astype(str)

# 2. RIMOZIONE DEGLI "A CAPO"
reviews['review_comment_title'] = reviews['review_comment_title'].replace(r'\r|\n', ' ', regex=True)
reviews['review_comment_message'] = reviews['review_comment_message'].replace(r'\r|\n', ' ', regex=True)

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print("Integrità referenziale e pulizia recensioni completata.")

Integrità referenziale e pulizia recensioni completata.


In [ ]:
# ESPORTAZIONE SICURA PER MARIADB
out_path = base_path + "Cleaned_Data/"
if not os.path.exists(out_path):
    os.makedirs(out_path)

# Aggiungiamo na_rep='\\N' a tutti i file per gestire correttamente i NULL su MariaDB
customers.to_csv(out_path + "customers_clean.csv", index=False, na_rep='\\N')
sellers.to_csv(out_path + "sellers_clean.csv", index=False, na_rep='\\N')
order_items.to_csv(out_path + "order_items_clean.csv", index=False, na_rep='\\N')
translation.to_csv(out_path + "translation_clean.csv", index=False, na_rep='\\N')

reviews.to_csv(out_path + "reviews_clean.csv", index=False, quoting=csv.QUOTE_ALL, quotechar='"', escapechar='\\', encoding='utf-8')
orders.to_csv(out_path + "orders_clean.csv", index=False, encoding='utf-8')
products.to_csv(out_path + "products_clean.csv", index=False, quoting=csv.QUOTE_ALL, quotechar='"', encoding='utf-8', lineterminator='\n', na_rep='\\N')

print("Tutti i file puliti sono stati salvati con supporto NULL nativo in:", out_path)

Tutti i file puliti sono stati salvati con supporto NULL nativo in: /content/drive/MyDrive/DataWerehouse_Project/Cleaned_Data/
